# 01 — RAG foundations: evidence before answers

## Scenario: Harborline Support

Support needs an assistant that can answer internal escalation questions from an approved mini-corpus. This notebook establishes the system boundary before adding embeddings or a model: retrieval chooses evidence; application policy decides whether evidence is sufficient; generation may only summarize that evidence.

Read the companion [What is RAG?](../../../docs/what-is-rag.md) guide for the broader lifecycle and references.

## The online RAG lifecycle

```text
authorized source -> parse/chunk -> index
question + identity -> filter -> retrieve -> rank -> bounded context
bounded context -> cite-supported answer OR abstain
```

A model does not become a database when it sees a prompt. Reliability depends on source quality, authorization, retrieval, evidence selection, citations, and an abstention policy.

In [ ]:
from collections import Counter
import re

corpus = [
    {'id': 'runbook-17', 'text': 'For checkout errors, correlate the 08:42 deployment with payment dependency latency before proposing rollback.', 'source': 'runbooks/checkout.md'},
    {'id': 'policy-04', 'text': 'After a confirmed incident, enterprise customers receive a status update within 30 minutes.', 'source': 'policies/sla.md'},
    {'id': 'guide-02', 'text': 'Use the health endpoint to inspect service and dependency availability.', 'source': 'guides/health.md'},
]

def tokens(text):
    return re.findall(r'[a-z0-9]+', text.lower())

def retrieve(question, top_k=2):
    query_terms = Counter(tokens(question))
    scored = []
    for document in corpus:
        score = sum((query_terms & Counter(tokens(document['text']))).values())
        if score:
            scored.append((document, score))
    return sorted(scored, key=lambda pair: (-pair[1], pair[0]['id']))[:top_k]

question = 'Checkout is slow after a deployment. What should support investigate?'
hits = retrieve(question)
[(document['id'], score, document['source']) for document, score in hits]

## Evidence is not an answer

A retrieval score is a ranking signal, not a confidence probability. Make the policy explicit: return a cited response only when evidence meets a transparent condition; otherwise abstain or ask a clarifying question. The rule below is intentionally simple and deterministic so you can inspect a failure before replacing it with evaluated relevance/groundedness checks.

In [ ]:
def answer_from_evidence(question, minimum_score=2):
    hits = retrieve(question)
    if not hits or hits[0][1] < minimum_score:
        return {'status': 'abstain', 'reason': 'insufficient-evidence', 'citations': []}
    citations = [{'id': document['id'], 'source': document['source']} for document, _ in hits]
    return {
        'status': 'answer',
        'answer': 'Investigate dependency latency and correlate it with the deployment before proposing rollback.',
        'citations': citations,
    }

print(answer_from_evidence(question))
print(answer_from_evidence('Which planet has rings?'))
assert answer_from_evidence('Which planet has rings?')['status'] == 'abstain'

## Experiments and checkpoint

1. Add a document with an exact error code and compare it with a paraphrase; why may lexical retrieval treat them differently?
2. Increase `top_k`; observe that more context is not automatically stronger evidence.
3. Delete the runbook and explain why a fluent model response would no longer be grounded.
4. Add tenant metadata and apply a filter before `retrieve`, then prove a cross-tenant text match never becomes a citation.

Next: [First local RAG](../02-first-local-rag/README.md) and the connected [Harborline notebook](../../../notebooks/beginner/01_first_local_rag.ipynb).